# Chapter 41 — Testing Stochastic Software

**Companion to *Pi Agents*** · [`Pi Agents` chapter 41](https://programmer.ie/books/pi/41-chapter/)

| | |
|---|---|
| Chapter | `41-chapter.md` · weight 41 · `/books/pi/41-chapter/` |
| Notebook | `notebooks/pi/41-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

How far apart are pass@1 and pass^5, and what does the gap hide?

The chapter's point: an agent is stochastic. **pass@1** (the average over single
runs) hides run-to-run reliability. **pass^k** — succeeded on all k independent
runs — exposes the gap. On a simulated 85% step across 40 tasks and 5 runs, the
book reproduces `pass@1 = 0.86` and `pass^5 = 0.45`.


## What this notebook establishes

- **Level 1 (scripted)**: a scripted response drives the code deterministically — it tests the code around the model.
- **Level 2 (structural)**: assert on the event stream, not on the wording — `tool:submit:ok`, one `turn_start`, one request.
- **Level 3 (evaluation)**: `pass@1 = 0.86` and `pass^5 = 0.45` are reproduced from a seeded 85% step over 40 tasks × 5 runs.
- A **thrown step counts as a failed run** and does not crash the evaluation.

## What this notebook does **not** establish

- **The 'forty tasks' are one task repeated forty times** — the figure is about the arithmetic, not a real workload.
- **The figures describe a simulation, not any real model** (`ch41-lim2`).
- **`pass^k` provenance is cited, not verified** in this notebook.
- **Level 3 with a real model is optional live work** — it needs a credential and costs money.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** + Reproduce — the chapter's own `evaluate()` over the seeded simulation; the real-model level 3 is absent.
* **Evidence scope:** `in_process_runtime`. The evaluation arithmetic is the chapter's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(41))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/41-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: levels 1 and 2, then the question level 3 answers

In [3]:
TESTING = pinb.driver_source("ch41-testing.ts")
tst = pinb.driver(TESTING, label="ch41-testing", timeout=420)
pk = tst["passK"]
print(pinb.md_table(
    ["metric", "value"],
    [["pass@1 (average over single runs)", pk["passAt1"]],
     ["pass^5 (succeeded on all 5 runs)", pk["passPowK"]]],
))

| metric | value |
|---|---|
| pass@1 (average over single runs) | 0.86 |
| pass^5 (succeeded on all 5 runs) | 0.45 |


In [4]:
pinb.show_checks([
    pinb.check("pass@1 reproduces 0.86 on the seeded 85% simulation",
               pk["passAt1"] == 0.86, f"{pk['passAt1']}"),
    pinb.check("pass^5 reproduces 0.45",
               pk["passPowK"] == 0.45, f"{pk['passPowK']}"),
    pinb.check("the gap is large: pass^5 is 0.4 below pass@1",
               pk["passPowK"] < pk["passAt1"] - 0.35, "the average hides reliability"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------------  -----------------------------
PASS  pass@1 reproduces 0.86 on the seeded 85% simulation  0.86
PASS  pass^5 reproduces 0.45  0.45
PASS  the gap is large: pass^5 is 0.4 below pass@1  the average hides reliability

3/3 assertions held.


In [5]:
gap = pk["passAt1"] - pk["passPowK"]
print(f"The gap: pass@1 - pass^5 = {gap:.2f}")
print()
print("What the gap hides: a run that is right 85% of the time on average")
print("succeeds on all 5 independent runs less than half the time.")

The gap: pass@1 - pass^5 = 0.41

What the gap hides: a run that is right 85% of the time on average
succeeds on all 5 independent runs less than half the time.


## A thrown step is a failed run, not a crash

In [6]:
ts = tst["thrownStep"]
print(pinb.md_table(
    ["check", "value"],
    [["runs completed after the throw", ts["calls"]],
     ["pass@1 over the two tasks", round(ts["passAt1"], 3)],
     ["pass^5", ts["passPowK"]]],
))

| check | value |
|---|---|
| runs completed after the throw | 5 |
| pass@1 over the two tasks | 0.4 |
| pass^5 | 0 |


In [7]:
pinb.show_checks([
    pinb.check("the evaluation kept going after the throw",
               ts["calls"] == 5, f"{ts['calls']} calls"),
    pinb.check("a thrown run counts as a failure",
               ts["passPowK"] == 0, "pass^5 = 0"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------  ----------
PASS  the evaluation kept going after the throw  5 calls
PASS  a thrown run counts as a failure  pass^5 = 0

2/2 assertions held.


## The chapter's own tests

All four testing tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [8]:
PATTERNS = 'ch41-testing/testing.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch41-testing/testing.test.ts', show="level")

$ node --test ch41-testing/testing.test.ts
  PASS  4 passed, 0 failed, 0 skipped  in 437 ms

  tests matching 'level':
    ok   level 1: a scripted response drives the code deterministically
    ok   level 2: assert on the event stream, not on the wording
    ok   level 3: pass^k exposes what the average hides
    ok   level 3: a thrown step counts as a failed run and does not crash the evaluation


## Your turn: predict, then run

**Predict first.** The simulation is seeded with 42. What happens to pass@1 and
pass^5 if you change the seed to 0 — do the pinned figures move?

**Then change one input.** Make the simulated model 95% reliable instead of 85%.
Which metric moves faster, pass@1 or pass^5?

**Predict the boundary.** Level 3 with a *real* model is the only level that
says anything about the model. What would it cost, and what must be frozen before
a run?

In [9]:
print("Predictions:")
print("  1. A different seed moves both figures - the pinned values are for seed 42.")
print("  2. pass^5 moves faster: it is a power, so a small change in p compounds.")
print("  3. Real-model level 3 needs a credential and a frozen contract (planning/real-model-plan.md).")
print()
print("Observed above:")
print(f"  pass@1={pk['passAt1']}, pass^5={pk['passPowK']}")
print(f"  thrown step: calls={ts['calls']}, pass^5={ts['passPowK']}")
print()
assert pk["passAt1"] == 0.86 and pk["passPowK"] == 0.45
assert ts["calls"] == 5 and ts["passPowK"] == 0
print("held: pass@1 = 0.86, pass^5 = 0.45; a thrown step is a failed run, not a crash")

Predictions:
  1. A different seed moves both figures - the pinned values are for seed 42.
  2. pass^5 moves faster: it is a power, so a small change in p compounds.
  3. Real-model level 3 needs a credential and a frozen contract (planning/real-model-plan.md).

Observed above:
  pass@1=0.86, pass^5=0.45
  thrown step: calls=5, pass^5=0

held: pass@1 = 0.86, pass^5 = 0.45; a thrown step is a failed run, not a crash


## Interpretation

Three levels of testing, and what each establishes:

| Level | What it does | What it cannot show |
|---|---|---|
| **1. Scripted** | Drives the code deterministically | Anything about a real model |
| **2. Structural** | Asserts on the event stream | Wording; model quality |
| **3. Evaluation** | Measures pass rates over repeated runs | Anything real, until the model is real |

The arithmetic: `pass@1 = 0.86` and `pass^5 = 0.45` on a simulated 85% step.
`pass@1` is the average; `pass^k` requires success on all k runs. The gap is
reliability, and it is the number a production decision should look at.

Practical rules:
1. **Use level 1 to test your code** — determinism around a model is scriptable.
2. **Use level 2 to pin structure** — events and request counts, not words.
3. **Use level 3 to make claims about reliability** — and read `pass^k`, not the average.
4. **A thrown run is a failure** — count it, do not crash the sheet.
5. **Real-model evaluation is separate** — a contract and a credential, never a cell.

## Sources, execution mode and limitations

**Execution mode:** **Run** + Reproduce — the chapter's own `evaluate()` over the seeded simulation; the real-model level 3 is absent.

**Evidence scope:** `in_process_runtime`. The evaluation arithmetic is the chapter's.

### What was read or run

- **Ran** `drivers/ch41-testing.ts`: the seeded 40×5 evaluation, reproducing pass@1 = 0.86, pass^5 = 0.45, and the thrown-step case.
- **Ran** `ch41-testing/testing.test.ts` (4 tests).
- **Read** `examples/evidence.json`, chapter 41 rows (4 claims).

### Limitations

- **The 'forty tasks' are one task repeated forty times** — the figure is about the arithmetic.
- **The figures describe a simulation, not any real model** (`ch41-lim2`).
- **`pass^k` provenance is cited, not verified**.

### Optional work that was NOT run

- Real-model level 3 evaluation — needs a credential, a frozen contract, and costs money.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
